In [1]:
import pandas as pd

df = pd.read_csv("../Week 2/cleaned_dataset.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())

Dataset shape: (9, 9)

Columns:
['sale_id', 'customer_name', 'product', 'category', 'quantity', 'price', 'city', 'sale_date', 'discount']


,sale_id,customer_name,product,category,quantity,price,city,sale_date,discount
0,1,Arun,Laptop,Electronics,1,33000.0,Bangalore,2026-08-01,10.0
1,3,Rahul,Chair,Furniture,1,4500.0,Mumbai,2026-08-05,10.0
2,4,Ananya,Keyboard,Electronics,1,2500.0,Bangalore,2026-08-07,8.0
3,5,Kiran,T-Shirt,Clothing,3,1200.0,Hyderabad,2026-08-10,15.0
4,6,Asha,Desk,Furniture,1,7000.0,Chennai,2026-08-12,10.0


In [2]:
# 1. COMPLETENESS
missing_values = df.isnull().sum()
total_missing = missing_values.sum()

print("Missing values by column:")
print(missing_values)
print("\nTotal missing values:", total_missing)


# 2. ACCURACY
print("\nAccuracy checks:")

print("Invalid quantity:", (df["quantity"] <= 0).sum())
print("Invalid price:", (df["price"] < 0).sum())
print("Invalid discount:", ((df["discount"] < 0) | (df["discount"] > 100)).sum())


# 3. CONSISTENCY
print("\nConsistency checks:")

print("Duplicate sale IDs:", df["sale_id"].duplicated().sum())
print("Duplicate rows:", df.duplicated().sum())

print("\nUnique categories:")
print(df["category"].unique())

print("\nUnique cities:")
print(df["city"].unique())

Missing values by column:
sale_id          0
customer_name    0
product          0
category         0
quantity         0
price            0
city             0
sale_date        0
discount         0
dtype: int64

Total missing values: 0

Accuracy checks:
Invalid quantity: 0
Invalid price: 0
Invalid discount: 0

Consistency checks:
Duplicate sale IDs: 0
Duplicate rows: 0

Unique categories:
<StringArray>
['Electronics', 'Furniture', 'Clothing']
Length: 3, dtype: str

Unique cities:
<StringArray>
['Bangalore', 'Mumbai', 'Hyderabad', 'Chennai', 'Delhi']
Length: 5, dtype: str


In [3]:
# 4. TIMELINESS
df["sale_date"] = pd.to_datetime(df["sale_date"])

today = pd.Timestamp.today().normalize()

future_dates = (df["sale_date"] > today).sum()

print("Future-dated records:", future_dates)
print("Date range:", df["sale_date"].min(), "to", df["sale_date"].max())


# 5. UNIQUENESS
unique_sale_ids = df["sale_id"].nunique()
total_records = len(df)

print("\nUnique sale IDs:", unique_sale_ids)
print("Total records:", total_records)
print("All sale IDs unique:", unique_sale_ids == total_records)


# 6. VALIDITY
valid_categories = ["Electronics", "Furniture", "Clothing"]

valid_category_count = df["category"].isin(valid_categories).sum()

print("\nValid category records:", valid_category_count)
print("Total records:", total_records)
print("All categories valid:", valid_category_count == total_records)

print("\nSale ID data type:", df["sale_id"].dtype)
print("Quantity data type:", df["quantity"].dtype)
print("Price data type:", df["price"].dtype)

Future-dated records: 0
Date range: 2026-08-01 00:00:00 to 2026-08-22 00:00:00

Unique sale IDs: 9
Total records: 9
All sale IDs unique: True

Valid category records: 9
Total records: 9
All categories valid: True

Sale ID data type: int64
Quantity data type: int64
Price data type: float64


In [4]:
# DAMA-DMBOK Data Quality Scorecard

scorecard = pd.DataFrame({
    "Dimension": [
        "Completeness",
        "Accuracy",
        "Consistency",
        "Timeliness",
        "Uniqueness",
        "Validity"
    ],
    "Score": [
        100,
        100,
        100,
        100,
        100,
        100
    ],
    "Status": [
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS"
    ]
})

display(scorecard)

,Dimension,Score,Status
0,Completeness,100,PASS
1,Accuracy,100,PASS
2,Consistency,100,PASS
3,Timeliness,100,PASS
4,Uniqueness,100,PASS
5,Validity,100,PASS


In [1]:
import great_expectations as gx

print("Great Expectations version:", gx.__version__)

Great Expectations version: 0.18.22


In [5]:
import great_expectations as gx

# Create a Great Expectations context
context = gx.get_context()

# Create Pandas datasource
datasource = context.sources.add_pandas(name="sales_datasource")

# Create data asset
data_asset = datasource.add_dataframe_asset(name="sales_data")

# Create batch request using our dataframe
batch_request = data_asset.build_batch_request(dataframe=df)

# Create an expectation suite
suite_name = "sales_data_quality_suite"
context.add_expectation_suite(expectation_suite_name=suite_name)

# Create validator
validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite_name=suite_name
)

# 1. Sale ID should not be null
validator.expect_column_values_to_not_be_null("sale_id")

# 2. Customer name should not be null
validator.expect_column_values_to_not_be_null("customer_name")

# 3. Quantity should be greater than 0
validator.expect_column_values_to_be_between(
    "quantity",
    min_value=1
)

# 4. Price should be greater than or equal to 0
validator.expect_column_values_to_be_between(
    "price",
    min_value=0
)

# 5. Discount should be between 0 and 100
validator.expect_column_values_to_be_between(
    "discount",
    min_value=0,
    max_value=100
)

# 6. Sale IDs should be unique
validator.expect_column_values_to_be_unique("sale_id")

# 7. Category should contain only valid categories
validator.expect_column_values_to_be_in_set(
    "category",
    ["Electronics", "Furniture", "Clothing"]
)

# 8. Sale date should not be null
validator.expect_column_values_to_not_be_null("sale_date")

# Save the expectation suite
validator.save_expectation_suite(discard_failed_expectations=False)

print("✅ 8 Great Expectations checks created successfully!")

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

✅ 8 Great Expectations checks created successfully!


In [4]:
import pandas as pd

df = pd.read_csv("../Week 2/cleaned_dataset.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (9, 9)


In [6]:
# Create a copy so our original cleaned dataset stays safe
test_df = df.copy()

# Inject 3 intentional data-quality errors
test_df.loc[0, "quantity"] = 0              # Error 1: invalid quantity
test_df.loc[1, "discount"] = 150            # Error 2: invalid discount
test_df.loc[2, "sale_id"] = test_df.loc[3, "sale_id"]  # Error 3: duplicate sale ID

print("3 intentional data-quality errors injected.")
print(test_df.loc[[0, 1, 2, 3]])

3 intentional data-quality errors injected.
   sale_id customer_name   product     category  quantity    price       city  \
0        1          Arun    Laptop  Electronics         0  33000.0  Bangalore   
1        3         Rahul     Chair    Furniture         1   4500.0     Mumbai   
2        5        Ananya  Keyboard  Electronics         1   2500.0  Bangalore   
3        5         Kiran   T-Shirt     Clothing         3   1200.0  Hyderabad   

    sale_date  discount  
0  2026-08-01      10.0  
1  2026-08-05     150.0  
2  2026-08-07       8.0  
3  2026-08-10      15.0  


In [7]:
# Create a batch using the intentionally bad dataset
bad_batch_request = data_asset.build_batch_request(dataframe=test_df)

# Create a validator for the bad data
bad_validator = context.get_validator(
    batch_request=bad_batch_request,
    expectation_suite_name=suite_name
)

# Validate the data
validation_result = bad_validator.validate()

print("Validation complete!")
print("Overall success:", validation_result["success"])

# Show failed expectations
print("\nFailed expectations:")
for result in validation_result["results"]:
    if not result["success"]:
        print(
            "-",
            result["expectation_config"]["expectation_type"]
        )

Calculating Metrics:   0%|          | 0/35 [00:00<?, ?it/s]

Validation complete!
Overall success: False

Failed expectations:
- expect_column_values_to_be_unique
- expect_column_values_to_be_between
- expect_column_values_to_be_between


In [8]:
# Create an HTML report from the validation result
html_report = validation_result["results"]

print("Validation results captured successfully!")
print("Total expectations checked:", len(html_report))
print("Failed expectations:", sum(1 for r in html_report if not r["success"]))

Validation results captured successfully!
Total expectations checked: 8
Failed expectations: 3


In [9]:
# Save the validation result as a JSON file
import json

with open("gx_validation_report.json", "w") as f:
    json.dump(validation_result, f, indent=4, default=str)

print("✅ GX validation report saved successfully!")
print("File: gx_validation_report.json")

✅ GX validation report saved successfully!
File: gx_validation_report.json


In [10]:
pii_classification = {
    "sale_id": "Non-PII",
    "customer_name": "PII",
    "product": "Non-PII",
    "category": "Non-PII",
    "quantity": "Non-PII",
    "price": "Non-PII",
    "city": "Potentially sensitive location data",
    "sale_date": "Non-PII",
    "discount": "Non-PII"
}

for column, classification in pii_classification.items():
    print(f"{column}: {classification}")

sale_id: Non-PII
customer_name: PII
product: Non-PII
category: Non-PII
quantity: Non-PII
price: Non-PII
city: Potentially sensitive location data
sale_date: Non-PII
discount: Non-PII


In [11]:
# Final Data Quality Scorecard

scorecard = pd.DataFrame({
    "Dimension": [
        "Completeness",
        "Accuracy",
        "Consistency",
        "Timeliness",
        "Uniqueness",
        "Validity"
    ],
    "Score": [100, 100, 100, 100, 100, 100],
    "Status": [
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS"
    ],
    "Recommendation": [
        "Continue monitoring missing values",
        "Validate values against business rules",
        "Maintain consistency across datasets",
        "Monitor data refresh SLA",
        "Enforce unique sale IDs",
        "Validate formats and allowed values"
    ]
})

display(scorecard)

,Dimension,Score,Status,Recommendation
0,Completeness,100,PASS,Continue monitoring missing values
1,Accuracy,100,PASS,Validate values against business rules
2,Consistency,100,PASS,Maintain consistency across datasets
3,Timeliness,100,PASS,Monitor data refresh SLA
4,Uniqueness,100,PASS,Enforce unique sale IDs
5,Validity,100,PASS,Validate formats and allowed values


In [ ]:
# Generate a simple HTML validation report

from pathlib import Path
import html

failed_results = [
    r for r in validation_result["results"]
    if not r["success"]
]

rows = ""

for r in validation_result["results"]:
    expectation = r["expectation_config"]["expectation_type"]
    status = "PASS" if r["success"] else "FAIL"
    rows += f"""
    <tr>
        <td>{html.escape(expectation)}</td>
        <td>{status}</td>
    </tr>
    """

report_html = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Great Expectations Validation Report</title>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 40px; }}
        table {{ border-collapse: collapse; width: 100%; }}
        th, td {{ border: 1px solid #ccc; padding: 10px; text-align: left; }}
        th {{ background: #f2f2f2; }}
    </style>
</head>
<body>
    <h1>Great Expectations Validation Report</h1>

    <h2>Summary</h2>
    <p><strong>Total expectations:</strong> {len(validation_result["results"])}</p>
    <p><strong>Failed expectations:</strong> {len(failed_results)}</p>
    <p><strong>Overall success:</strong> {validation_result["success"]}</p>

    <h2>Expectation Results</h2>
    <table>
        <tr>
            <th>Expectation</th>
            <th>Status</th>
        </tr>
        {rows}
    </table>
</body>
</html>
"""

report_path = Path("gx_validation_report.html")
report_path.write_text(report_html, encoding="utf-8")

print("✅ HTML validation report created!")
print("File:", report_path)